# Apache Avro

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-interview-questions/blob/master/notebooks/topics/avro.ipynb)

*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

## What is Apache Avro?

An open source project which offers data serialization as well as data exchange services, originally for Apache Hadoop, is what we call Apache Avro. It is possible to use these services together or independently. Programs can efficiently serialize data into files or into messages with the serialization service, and data storage is very compact and efficient because the data definition (the schema) is written in JSON while the data itself is stored in a binary format.

An Avro schema is a JSON document that declares a record name, a namespace, and typed fields:

```json
{
  "type": "record",
  "name": "OrderEvent",
  "namespace": "com.example.orders",
  "doc": "One order lifecycle event emitted by the orders service.",
  "fields": [
    {"name": "order_id", "type": "string"},
    {"name": "customer_id", "type": "string"},
    {"name": "status", "type": {"type": "enum", "name": "OrderStatus", "symbols": ["CREATED", "PAID", "CANCELLED", "REFUNDED"]}},
    {"name": "amount", "type": {"type": "bytes", "logicalType": "decimal", "precision": 12, "scale": 2}},
    {"name": "currency", "type": "string", "default": "USD"},
    {"name": "event_time", "type": {"type": "long", "logicalType": "timestamp-millis"}}
  ]
}
```

The schema is the contract: a writer always serializes with a writer schema, and a reader always resolves the bytes with a reader schema. In an Avro Object Container File (`.avro`) the writer schema plus a sync marker are stored in the header, so the file is self-describing. In Kafka, the schema usually travels separately through a schema registry and only a small schema ID is embedded in each message.

Key interview point: Avro is *schema-on-write* with *schema evolution*. Unlike JSON there is no field name repeated for every record, which is why Avro payloads are typically several times smaller than equivalent JSON.

Think of a record as a row of values where the types live in a separate contract, not in the bytes. The writer lays the fields down in schema order with no field names and no type tags. A `long` becomes a zig-zag varint that takes 1 byte for small values and up to 10 bytes for huge ones. A `string` becomes its byte length as a varint followed by UTF-8 bytes. The reader needs the writer schema to know where each value starts and ends.

Worked example. The record `{"order_id": "O-1001", "status": "PAID", "amount": 42.50}` costs about 1 byte of length plus 6 characters for the id, 1 byte for the enum branch index, and 8 bytes for the double, so roughly 16 bytes of payload. The JSON form is about 60 bytes. Common mistake: assuming a reader can decode without the writer schema. That is true for JSON, not for Avro.

## What Avro Offers?

Avro offers:
+ Rich data structures (records, arrays, maps, unions, enums, fixed).
+ A compact, fast, binary data format.
+ A container file (`.avro`) to store persistent data with the schema embedded.
+ Remote procedure calls (RPC) through its own protocol definition.
+ Schema evolution through writer/reader schema resolution.

Example: the same order event as JSON versus Avro on the wire.

```text
JSON (field names repeated in every record, ~220 bytes):
{"order_id":"O-1001","customer_id":"C-7","status":"PAID","amount":42.50,"currency":"USD","event_time":"2026-10-04T15:03:02Z"}
```


That size difference compounds at scale: 1 billion events per day at 220 bytes of JSON is roughly 220 GB/day before compression; the same stream in Avro is closer to 35 GB/day.

Inside an object container file the header holds the magic bytes `Obj` plus version 1, a metadata map, and a 16-byte random sync marker. The metadata map carries `avro.schema` (the writer schema as JSON) and `avro.codec`. After the header the file is a sequence of blocks. Each block is a record count, a byte size, the encoded records, and another copy of the sync marker. A reader can seek to any sync marker, so one file splits across parallel tasks.

Avro RPC is the second service. A protocol file (`.avpr`) declares messages, and each message has a request schema and a response schema. The handshake swaps protocol fingerprints so both sides agree on the schema before any payload moves. Common mistake: treating an RPC protocol and a record schema as the same artifact. A data file needs only the record schema.

## State some Key Points about Apache Avro?

Some key points are:
+ Avro is a data serialization system.
+ It uses JSON-based schemas.
+ It uses RPC calls to send data.
+ During data exchange the schema is sent (or referenced), not repeated per record.
+ Avro is language-neutral: Java, Python, C, C++, C#, Go, Ruby, and others can read the same data.
+ Field identity is by name and (in newer schema designs) by aliases; Avro does not use numeric field IDs the way Protocol Buffers does.

Avro is schema-on-write. The producer commits to a schema when it writes, and every consumer resolves those bytes against its own reader schema. That split is what makes evolution possible: the writer never needs to know the reader. Common mistake: expecting a consumer to see a new field it never declared. The reader schema decides which fields exist for that consumer.

## Explain Avro schemas?

Mainly, Avro heavily depends on its schema. Basically, it permits every datum to be written with no prior knowledge of the schema at read time, because the writer schema is available in the file header or from the schema registry. We can say Avro serializes fast and the data resulting after serialization is least in size with schemas.

Every Avro schema also has a canonical form used to detect whether two schemas are textually the same. Two schemas that differ only in whitespace, field order in JSON, `doc`, or `aliases` produce the same parsing canonical form, which is why a schema registry can tell you "this is the same schema, reuse it" instead of registering a new version.

```json
{
  "type": "record",
  "name": "PageView",
  "namespace": "com.example.web",
  "fields": [
    {"name": "user_id", "type": "string"},
    {"name": "url", "type": "string"},
    {"name": "ts", "type": {"type": "long", "logicalType": "timestamp-millis"}},
    {"name": "props", "type": {"type": "map", "values": "string"}, "default": {}},
    {"name": "experiments", "type": {"type": "array", "items": "string"}, "default": []}
  ]
}
```

Interview note: defaults are not optional metadata. A field with no `default` is required, and a union that includes `null` should normally declare `"default": null` so old readers of old data do not fail.

A schema is one of four JSON shapes: a string primitive such as `"string"`, an object for a complex or logical type, a JSON array for a union, or a reference to an already declared named type. Records, enums, and fixed types are named; arrays, maps, and unions are anonymous. A full name is `namespace.name`, and both parts must start with a letter or underscore.

The parsing canonical form strips whitespace, `doc`, and `aliases`, orders object keys, and normalizes defaults. Registries use it to answer "have I seen this exact schema". Common mistake: assuming a `doc` edit creates a new version. It produces the same canonical form, so the registry usually deduplicates it.

## What do you mean by schema declaration?

In JSON, a Schema is represented by one of:
+ A JSON string, such as `"string"` or `"com.example.orders.OrderEvent"`.
+ A JSON object, such as `{"type": "record", "name": "Order", "fields": [...]}`.
+ `{"type": "typename", ...attributes...}` for a type with extra attributes.
+ A JSON array, which declares a union, such as `["null", "string"]`.

Examples:

```json
"string"
```

```json
{"type": "array", "items": "long"}
```

```json
["null", {"type": "record", "name": "Address", "fields": [
  {"name": "city", "type": "string"},
  {"name": "postcode", "type": ["null", "string"], "default": null}
]}]
```

A named type is declared once and then referenced by its fully qualified name, for example `"com.example.orders.OrderEvent"`.

Declare each named type once, then refer to it by full name. A record may reference itself, which models a linked list or a tree. The same full name cannot be declared twice inside one schema. Common mistake: re-declaring a nested record in a second field instead of referencing it. That creates two types with the same name and fails validation.

## What is the way of creating Avro Schemas?

In the format "lightweight text-based data interchange", JavaScript Object Notation (JSON), the Avro schema gets created. We can make it in various ways:
+ A JSON string
+ A JSON object
+ A JSON array (union)

In practice there are four ways to produce a schema:

```text
1. Hand-write the JSON schema (small, stable events).
2. Generate it from code: Java SpecificData / Avro Maven plugin, or Python avro.schema.
3. Reflect it from an existing type: Java ReflectData, or Pydantic/dataclass converters.
4. Convert from another contract: Avro IDL (.avdl) or a SQL DDL mapping in a CDC connector.
```

Avro IDL is a higher-level syntax when hand-writing JSON gets unwieldy:

```text
@namespace("com.example.orders")
protocol Orders {
  enum OrderStatus { CREATED, PAID, CANCELLED, REFUNDED }
  record OrderEvent {
    string order_id;
    string customer_id;
    OrderStatus status;
    decimal(12,2) amount;
    union { null, string } coupon_code = null;
  }
}
```

Compile it with `avro-tools idl2schemata orders.avdl out/` or `avro-tools compile protocol orders.avdl out.jar` in a build step.

In real projects the JSON schema is generated or reviewed rather than hand-typed. Avro IDL (`.avdl`) compiles to `.avsc` through `avro-tools idl2schemata` or the Maven plugin in a build step. Keep the schema in git next to the code, and register the same bytes in the registry. Common mistake: letting the registry and the repository drift, so a rollback replays a schema nobody can find.

## Name some primitive types of Data Types which Avro Supports.

Avro supports a wide range of primitive datatypes:
+ Null: no value
+ Boolean: a binary value
+ Int: 32-bit signed integer
+ Long: 64-bit signed integer
+ Float: single precision (32-bit) IEEE 754 floating-point number
+ Double: double precision (64-bit) IEEE 754 floating-point number
+ Bytes: the sequence of 8-bit unsigned bytes
+ String: Unicode character sequence

Avro also defines logical types that annotate primitives with a meaning, while keeping the underlying primitive on the wire:

```json
{
  "type": "record",
  "name": "Payment",
  "fields": [
    {"name": "payment_id", "type": "string"},
    {"name": "amount", "type": {"type": "bytes", "logicalType": "decimal", "precision": 18, "scale": 2}},
    {"name": "created_at", "type": {"type": "long", "logicalType": "timestamp-micros"}},
    {"name": "value_date", "type": {"type": "int", "logicalType": "date"}},
    {"name": "latency_ms", "type": {"type": "int", "logicalType": "time-millis"}},
    {"name": "external_id", "type": {"type": "string", "logicalType": "uuid"}}
  ]
}
```

Interview caution: a reader that does not understand a logical type still reads the primitive, but it loses the meaning (a decimal becomes raw bytes, a timestamp becomes a long). Always confirm that the consumer engine supports the logical types you choose.

Logical types annotate a primitive: `decimal` on `bytes` or `fixed` with `precision` and `scale`, `date` on `int` (days since the epoch), `time-millis` on `int`, `time-micros` on `long`, `timestamp-millis` and `timestamp-micros` on `long`, `uuid` on `string`, and `duration` on a 12-byte `fixed`. The underlying primitive is still what travels on the wire.

Worked example: `{"type": "bytes", "logicalType": "decimal", "precision": 18, "scale": 2}` stores 42.50 as the unscaled integer 4250 in big-endian bytes. A reader that ignores logical types still reads bytes, but loses the meaning. Common mistake: putting `timestamp-millis` on an `int`. A 32-bit signed value runs out about 24 days after the epoch, so timestamps need `long`.

## Name some complex types of Data Types which Avro Supports.

There are six kinds of complex types which Avro supports:
+ Records
+ Enums
+ Arrays
+ Maps
+ Unions
+ Fixed

A single schema that exercises all of them:

```json
{
  "type": "record",
  "name": "Envelope",
  "namespace": "com.example.common",
  "fields": [
    {"name": "event_id", "type": "string"},
    {"name": "kind", "type": {"type": "enum", "name": "Kind", "symbols": ["ORDER", "PAYMENT", "REFUND"]}},
    {"name": "tags", "type": {"type": "array", "items": "string"}, "default": []},
    {"name": "attributes", "type": {"type": "map", "values": "string"}, "default": {}},
    {"name": "payload", "type": ["null", "com.example.orders.OrderEvent"], "default": null},
    {"name": "checksum", "type": {"type": "fixed", "name": "Md5", "size": 16}}
  ]
}
```

Points worth making in an interview:
+ A union is an ordered list of types; on write the branch is chosen by the value's type, and on read the reader's first matching branch wins.
+ A field default must match the first branch of a union, so put `null` first when you want `"default": null`; the union does not otherwise require `null` to be first.
+ Named complex types must be defined before they are referenced by name within the same schema.
+ Maps always have string keys; use an array of records if keys are not strings.
+ `fixed` is sized bytes with a name, used for checksums and keys.

Union rules: no union inside a union, and no two branches of the same unnamed type. On write the first branch that matches the value wins. On read the writer branch must exist in the reader union, or resolution fails. Put `null` first when the field is optional so `"default": null` is legal.

Worked example: `"type": ["null", "string"], "default": null` is an optional string. `"type": ["string", "null"], "default": null` is invalid because the default must match the first branch. An enum can declare a `default` symbol so a reader that lacks a new symbol still resolves. Common mistake: removing an enum symbol that old files still contain.

## Explain Thrift and Protocol Buffers and Avro?

The most competent libraries with Avro are Thrift and Protocol Buffers. The difference between them is: as per the need, Avro supports both dynamic and static types. Basically, to specify schemas and their types, Protocol Buffers and Thrift use Interface Definition Languages (IDLs). Avro is built in the Hadoop ecosystem, while Thrift and Protocol Buffers are not.

Additional trade-offs:

+ Protocol Buffers and Thrift use numeric field tags, so renames are free but tags must never be reused.
+ Avro uses field names, so a rename is a breaking change unless you add an `aliases` entry.
+ Avro embeds (or resolves) the full writer schema, which makes dynamic/self-describing reads easy but message headers slightly larger in registry-based setups.
+ Protocol Buffers' `oneof` maps naturally to an Avro union; Protobuf's strict `required` was removed in proto3, while Avro leans on defaults.

Interview framing: "All three are binary and schema-driven. Choose Avro when Kafka and Hadoop/Spark compatibility and schema-registry-driven evolution matter most; choose Protobuf for gRPC service APIs; choose Thrift when you already run a Thrift service stack."

On the wire, Protobuf and Thrift write a tag or field id before each value, so they can skip unknown fields by number and renames are free. Avro writes values in schema order with no tags and relies on the writer schema plus name or alias matching, so renames need aliases. JSON writes field names in every record and needs no schema at all, at the cost of size.

For service APIs, gRPC plus Protobuf is the common stack, and Avro RPC is rare outside Hadoop-era systems. For event streams, Avro plus a registry is common. Common mistake: choosing a format from a size benchmark alone and ignoring which registry, tooling, and client libraries the team already runs.

## Why Avro?

Some features where Avro differs from other systems are:
+ Dynamic typing.
+ Untagged data (no per-record field tags or field names).
+ No manually assigned field IDs.
+ Schema travels with the data, so any reader can decode it.

Avro therefore supports two usage styles:

In [ ]:
# ILLUSTRATIVE - requires fastavro: install to run, or read on.
# 1. Dynamic / generic: decode with the writer schema read from the file.
from fastavro import reader

with open("orders.avro", "rb") as src:
    for record in reader(src, reader_schema=None):
        print(record)  # dict, no generated class needed

In [ ]:
# 2. Static / generated: compile the schema to a Python class or Java class first.
# Java: avro-tools compile schema orders.avsc out/
# Python: dataclasses-avroschema or a hand-written dataclass plus fastavro schemaless writer

The dynamic path is what makes Avro pleasant for ETL: a generic job can consume any topic whose schema is registered, without shipping a generated class per topic.

Untagged fields keep the payload close to the useful bytes, but the schema becomes mandatory. Dynamic typing lets a generic reader process a topic it has never seen a generated class for. No field ids means field names are the contract, so an accidental rename is a breaking change. Common mistake: shipping a schemaless Avro payload to a consumer that cannot fetch the writer schema. The bytes become undecodable.

## What are best features of Apache Avro?

Some of the best features of Avro are:
+ Schema evolution
+ Untagged data
+ Language support
+ Transparent compression
+ Dynamic typing
+ Native support in MapReduce
+ Rich data structures

Compression and splitting are the two features interviewers dig into. Avro container files store compressed blocks plus a sync marker every N bytes, so a block is the unit of compression:

In [ ]:
# ILLUSTRATIVE - requires fastavro: install to run, or read on.
from fastavro import writer

with open("orders.avro", "wb") as out:
    writer(
        out,
        parsed_schema,
        events,
        codec="zstandard",          # null, deflate, snappy, zstandard, bzip2, xz
        sync_interval=64 * 1024,    # sync marker roughly every 64 KiB
    )

Because each block starts after a sync marker, a distributed reader can seek to a block boundary and split the file across Spark tasks, which is exactly why Avro is splittable while an opaque gzip file is not.

Block framing is what makes Avro files splittable. The writer buffers records, compresses the whole block as one unit, then writes the record count, the compressed size, the bytes, and a 16-byte sync marker. `sync_interval` controls how often the marker repeats. A distributed reader seeks to the next marker and starts a new task there, so the block is both the compression unit and the parallel read unit.

Codec names recorded in `avro.codec` are `null`, `deflate`, `snappy`, `bzip2`, `xz`, and `zstandard`. Because each block is compressed on its own, blocks decode independently. Common mistake: gzipping an `.avro` file from the outside. The outer gzip is one opaque stream, so the file loses splittability. Set the codec on the writer instead.

## Explain some advantages of Avro.

Pros of Avro are:
+ The smallest size.
+ It compresses block at a time; split table.
+ Maintained object structure.
+ Supports reading old data with a new schema.
+ Schema is machine-readable, so it can drive CI compatibility checks and documentation.

In [ ]:
# ILLUSTRATIVE - requires fastavro: install to run, or read on.
import io
from fastavro import writer, reader

buf = io.BytesIO()
writer(buf, parsed_schema, events)
raw = buf.getvalue()

# Self-describing: the writer schema travels in the header.
buf.seek(0)
print(reader(buf).writer_schema["name"])  # com.example.orders.OrderEvent (fastavro stores the full name)

Field names are paid once per file or once per registered schema, not once per record. For a topic with ten string fields, JSON repeats ten names in every message while Avro writes them in the header. The saving grows with the number of messages, not with the size of one message.

Common mistake: claiming Avro is always smaller and faster. When a query scans two columns out of fifty, a columnar format reads far less data, because Avro must decode whole rows.

## Explain some disadvantages of Avro.

Cons of Avro are:
+ It must use .NET 4.5, in the case of C# Avro, to make the best use of it.
+ Potentially slower serialization than hand-rolled binary formats.
+ In order to read/write data, you need a schema.
+ The binary payload is not human-readable, so debugging needs tooling such as `avro-tools tojson`.
+ Reusing one schema for many event types leads to wide records and poor evolution hygiene.

```bash
# Inspect an Avro file from the command line.
java -jar avro-tools.jar tojson --pretty orders.avro | head -3
java -jar avro-tools.jar getschema orders.avro
```

Avro needs a reader with a compatible schema. Container files are self-describing, but bare Kafka payloads are not. There is no column pruning, and debugging needs tooling because the bytes are not text. Common mistake: storing business data as schemaless Avro blobs with the schema living only in a producer's source code, so nobody can decode the archive later.

## Explain the term Serialization?

To transport the data over the network or to store it on some persistent storage, the process of translating data structures or object state into binary or textual form is what we call Serialization. In other words, serialization is also known as marshaling and deserialization is known as unmarshalling.

A complete fastavro round trip:

In [ ]:
# ILLUSTRATIVE - requires fastavro: install to run, or read on.
from decimal import Decimal
from fastavro import writer, reader, parse_schema

schema = parse_schema({
    "type": "record",
    "name": "OrderEvent",
    "namespace": "com.example.orders",
    "fields": [
        {"name": "order_id", "type": "string"},
        {"name": "customer_id", "type": "string"},
        {"name": "status", "type": "string"},
        {"name": "amount", "type": {"type": "bytes", "logicalType": "decimal", "precision": 12, "scale": 2}},
        {"name": "currency", "type": "string", "default": "USD"},
        {"name": "event_time", "type": {"type": "long", "logicalType": "timestamp-millis"}},
    ],
})

events = [
    {
        "order_id": "O-1001",
        "customer_id": "C-7",
        "status": "PAID",
        "amount": Decimal("42.50"),
        "currency": "USD",
        "event_time": 1791138182000,
    }
]

with open("orders.avro", "wb") as out:
    writer(out, schema, events)

with open("orders.avro", "rb") as src:
    for record in reader(src):
        print(record)

Serialization turns a value into bytes using the schema. The Avro encoding rules: `int` and `long` are zig-zag encoded and then written as base-128 varints; `float` is 4 bytes and `double` is 8 bytes little-endian; `boolean` is one byte; `null` is zero bytes; strings and bytes are a length varint followed by the payload; arrays and maps are a sequence of blocks, each block a count then its items, closed by a zero-count block.

In [ ]:
def zigzag(n):
    return (n << 1) ^ (n >> 63)   # 64-bit form; small negatives stay small


def varint_size(n):
    z = zigzag(n)
    return max(1, (z.bit_length() + 6) // 7)


for value in [0, -1, 63, 64, -64, 8191, 8192, 2**31 - 1, 2**63 - 1]:
    print(f"{value:>19} -> zigzag={zigzag(value):>20} varint bytes={varint_size(value)}")

assert varint_size(0) == 1
assert varint_size(-1) == 1
assert varint_size(64) == 2
assert varint_size(8192) == 3
assert varint_size(2**63 - 1) == 10

Common mistake: assuming varints make every number small. A value near 2^63 takes 10 bytes, and a random 64-bit long averages about 9. Zig-zag keeps small positive and negative numbers cheap, which is why `-1` costs one byte.

## Name some Avro Reference Apis?

The classes and methods which we use in the serialization, as well as deserialization, of Avro schemas are:

Java:
+ `SpecificDatumWriter`
+ `SpecificDatumReader`
+ `DataFileWriter`
+ `DataFileReader`
+ `Schema.Parser`
+ `GenericRecord`
+ `GenericData.Record`

Python and other languages:
+ `fastavro.writer`, `fastavro.reader`, `fastavro.parse_schema`, `fastavro.schemaless_writer`
+ `avro.schema.parse`, `avro.io.DatumReader`, `avro.io.DatumWriter`, `avro.datafile.DataFileReader`
+ `confluent_kafka.schema_registry.avro.AvroSerializer` / `AvroDeserializer` for Kafka

In [ ]:
# ILLUSTRATIVE - requires fastavro: install to run, or read on.
from fastavro import parse_schema, schemaless_writer, schemaless_reader
import io

schema = parse_schema({"type": "record", "name": "Ping", "fields": [{"name": "ts", "type": "long"}]})

buf = io.BytesIO()
schemaless_writer(buf, schema, {"ts": 1791138182000})   # no header, for Kafka payloads
buf.seek(0)
print(schemaless_reader(buf, schema))

Use `fastavro` for speed and dict-based records, or the reference `avro` package when you need generated classes and exact spec behavior. `parse_schema` validates and normalizes once; `schemaless_writer` writes a bare payload for Kafka; `writer` and `reader` handle container files. Common mistake: writing with `schemaless_writer` and reading with `reader`. The bare payload has no header, so the reader cannot find a schema.

## How to use Avro?

The workflow to use Avro is:
1. Create or obtain the schema.
2. Read the schema into your program, either by generating a class with a parser/library or by parsing it dynamically.
3. Serialize using the serialization API.
4. Deserialize using the deserialization API, supplying the writer schema if it is not embedded.

PySpark reads and writes Avro with the `spark-avro` package (`--packages org.apache.spark:spark-avro_2.12:3.5.1`):

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
orders = (
    spark.read
    .format("avro")
    .load("s3://company-raw/orders/")
)

orders.write.format("avro").mode("append").save("s3://company-curated/orders/")

Kafka producer with the Confluent serializer (schema registry assigns and stores the writer schema; the message carries a 5-byte magic byte + schema ID prefix):

In [ ]:
# ILLUSTRATIVE - requires confluent_kafka: install to run, or read on.
from confluent_kafka import Producer
from confluent_kafka.schema_registry import SchemaRegistryClient
from confluent_kafka.schema_registry.avro import AvroSerializer
from confluent_kafka.serialization import MessageField, SerializationContext

order_schema_str = """
{
  "type": "record",
  "name": "OrderEvent",
  "namespace": "com.example.orders",
  "fields": [
    {"name": "order_id", "type": "string"},
    {"name": "status", "type": "string"},
    {"name": "amount", "type": "double"}
  ]
}
"""

registry = SchemaRegistryClient({"url": "https://schema-registry.example.com"})
serializer = AvroSerializer(registry, order_schema_str)

producer = Producer({"bootstrap.servers": "kafka-1:9092,kafka-2:9092"})
producer.produce(
    topic="orders.events.v1",
    key="O-1001",
    value=serializer(
        {"order_id": "O-1001", "status": "PAID", "amount": 42.50},
        SerializationContext("orders.events.v1", MessageField.VALUE),
    ),
)
producer.flush()

The loop is: get the schema, serialize with it, and hand the same schema or its registry id to the consumer. With a registry the producer registers the schema and each message carries a magic byte plus a 4-byte schema id; the consumer fetches that id and resolves it against its own reader schema. Common mistake: generating the consumer class from a cached schema file instead of fetching the writer schema. When the producer evolves, decoding fails or silently drops fields.

## Explain sort order in brief?

There is a standard sort order for data in Avro which allows data written by one system to be efficiently sorted by another system. As sort order comparisons are sometimes the most frequent per-object operation, it can be an important optimization.

Every type has a defined ordering, and a schema can override it per field:

```json
{
  "type": "record",
  "name": "LeaderboardEntry",
  "fields": [
    {"name": "player_id", "type": "string"},
    {"name": "score", "type": "long", "order": "descending"},
    {"name": "updated_at", "type": {"type": "long", "logicalType": "timestamp-millis"}, "order": "ignore"}
  ]
}
```

+ Default ordering for records is field-by-field in declaration order, using each field's `order`.
+ `ascending` (default), `descending`, and `ignore` are the allowed values.
+ Unions order by branch index first, which is why `["null", "string"]` and `["string", "null"]` are different schemas.
+ Avro does not ship a distributed sort; the point is that two systems agree on *how* to compare, not that either sorts for you.

Avro defines a total order per type so a reducer or external sort can compare records without knowing the writer's intent. Records compare field by field in declaration order, arrays and maps compare element-wise, and unions compare by branch index first. A field's `order` attribute (`ascending`, `descending`, `ignore`) changes comparison only, never the encoding.

Common mistake: expecting `order` to reorder stored bytes. It does not. It only tells a sorting consumer how to compare, and `ignore` still writes and reads the field.

## What do you mean by Schema Resolution?

Whether from an RPC or a file, a reader of Avro data can always parse that data since its schema is offered. Yet it is possible that the schema may not be exactly the schema we expect, so for that purpose we use Schema Resolution.

Resolution rules that matter in interviews:

+ Fields are matched by name (or alias) between writer and reader schema.
+ A reader field with a default is filled in when the writer did not write it.
+ A writer field absent from the reader schema is skipped.
+ Numeric promotion is allowed in the widening direction: `int` to `long`, `float`, or `double`; `long` to `float` or `double`; `float` to `double`.
+ Union resolution matches the writer's branch against a reader branch; the writer's branch must be present in the reader union (unless it can be promoted).
+ An enum symbol unknown to the reader needs a reader-side default (`"default": "UNKNOWN"` on the enum) or resolution fails.

In [ ]:
# ILLUSTRATIVE - requires fastavro: install to run, or read on.
import io
from fastavro import parse_schema, schemaless_writer, schemaless_reader

writer_schema = parse_schema({
    "type": "record",
    "name": "Order",
    "fields": [{"name": "order_id", "type": "string"}],
})

reader_schema = parse_schema({
    "type": "record",
    "name": "Order",
    "fields": [
        {"name": "order_id", "type": "string"},
        {"name": "currency", "type": "string", "default": "USD"},
    ],
})

buf = io.BytesIO()
schemaless_writer(buf, writer_schema, {"order_id": "O-1001"})
buf.seek(0)

# Old bytes, new reader schema: 'currency' comes from the default.
print(schemaless_reader(buf, writer_schema, reader_schema))

Resolution compares the writer schema to the reader schema. Primitives must match, with widening allowed: `int` to `long`, `float`, or `double`; `long` to `float` or `double`; `float` to `double`; and `string` to `bytes` or back. Record fields match by name or alias. A reader field the writer never wrote is filled from its default. A writer field the reader does not declare is skipped. Enum symbols must exist in the reader unless the reader enum declares a `default`. Fixed types must have equal size.

In [ ]:
def field_index(schema):
    return {f["name"]: f for f in schema["fields"]}


def backward(old, new):
    """New schema can read data written with the old schema."""
    old_fields = field_index(old)
    for name, field in field_index(new).items():
        if name not in old_fields and "default" not in field:
            return False
    return True


def forward(old, new):
    """Old schema can read data written with the new schema."""
    new_fields = field_index(new)
    for name, field in field_index(old).items():
        if name not in new_fields and "default" not in field:
            return False
    return True


old = {"type": "record", "name": "Order", "fields": [
    {"name": "order_id", "type": "string"},
    {"name": "status", "type": "string"},
]}
add_optional = {"type": "record", "name": "Order", "fields": [
    {"name": "order_id", "type": "string"},
    {"name": "status", "type": "string"},
    {"name": "currency", "type": "string", "default": "USD"},
]}
add_required = {"type": "record", "name": "Order", "fields": [
    {"name": "order_id", "type": "string"},
    {"name": "status", "type": "string"},
    {"name": "currency", "type": "string"},
]}

for label, candidate in [("add optional", add_optional), ("add required", add_required)]:
    print(f"{label:13} BACKWARD={backward(old, candidate)} FORWARD={forward(old, candidate)}")

assert backward(old, add_optional) and forward(old, add_optional)        # FULL
assert not backward(old, add_required) and forward(old, add_required)    # FORWARD only
assert backward(old, old) and forward(old, old)

Common mistake: expecting a rename to resolve on its own. Field names are the identity in Avro, so a rename needs an alias on the reader side or it looks like one field dropped and another added.

## How does Avro schema evolution work?

Avro supports compatible schema changes when readers and writers can resolve differences. Common compatible changes include adding a field with a default value, removing a field that readers do not require, or widening some types depending on compatibility rules. Breaking changes include renaming a field without aliases or changing a type incompatibly.

Compatibility by change type:

| Change | Backward (new reader, old data) | Forward (old reader, new data) | Full |
|---|---|---|---|
| Add field with `default` | Yes | Yes | Yes |
| Add field without `default` | No | Yes | No |
| Remove field with `default` | Yes | Yes | Yes |
| Remove required field | Yes | No | No |
| Widen `int` to `long` | Yes | No | No |
| Rename field + add `aliases` | Yes | No | No |
| Change `string` to `int` | No | No | No |
| Reorder union branches | No | No | No |

Aliases are applied by rewriting the *writer's* schema using aliases from the *reader's* schema, so adding an alias to the new schema makes the rename backward compatible; it is not forward compatible unless the old reader also carries an alias for the new name (or the old field has a default).

Example: adding a field safely.

```json
{"name": "currency", "type": "string", "default": "USD"}
```

Because old messages do not contain `currency`, the default lets new readers handle old data. In production, schema compatibility should be enforced in CI or by a schema registry:

```bash
# Confluent Schema Registry: set and check the topic-level compatibility policy.
curl -s -X PUT -H "Content-Type: application/vnd.schemaregistry.v1+json" \
  --data '{"compatibility": "BACKWARD"}' \
  https://schema-registry.example.com/config/orders.events.v1

# CI gate: test a candidate schema before merge.
curl -s -X POST -H "Content-Type: application/vnd.schemaregistry.v1+json" \
  --data "{\"schema\": $(python -c 'import json;print(json.dumps(open("orders_v2.avsc").read()))')}" \
  https://schema-registry.example.com/compatibility/subjects/orders.events.v1-value/versions/latest
```

Upgrade order matters. If the policy is BACKWARD, upgrade consumers first, then producers. If it is FORWARD, do the reverse. If it is FULL, either order is safe. Every new field should be optional with a default, every rename needs a reader alias, and a removal should be deprecated for one release by giving the field a default before it disappears.

Worked example. v1 has `order_id` and `status`. v2 adds `currency` with default `USD` and renames `status` to `state` with `"aliases": ["status"]`. Old data decodes under v2 because `currency` falls back to its default and `state` matches the alias. New data decodes under v1 because the old reader skips `currency` and still finds `status`. That is FULL compatibility.

## What are Avro compatibility modes and how does a schema registry enforce them?

A schema registry stores versioned schemas per subject (usually `<topic>-value`) and rejects a new version that violates the subject's compatibility policy. The four modes interviewers expect:

+ `BACKWARD`: new schema can read data written with the previous schema. This is the default and the safest for consumers-first upgrades.
+ `FORWARD`: previous schema can read data written with the new schema. Safe when producers upgrade before consumers.
+ `FULL`: both directions. Restrictive but the safest for rolling deployments.
+ `NONE`: no checks. Use only for greenfield topics where you control both ends.

```yaml
# Debezium / Kafka Connect sink connector referencing the registry.
name: orders-avro-sink
config:
  connector.class: io.confluent.connect.s3.S3SinkConnector
  topics: orders.events.v1
  s3.bucket.name: company-curated
  format.class: io.confluent.connect.s3.format.avro.AvroFormat
  schema.compatibility: FULL
  value.converter: io.confluent.connect.avro.AvroConverter
  value.converter.schema.registry.url: https://schema-registry.example.com
  key.converter: org.apache.kafka.connect.storage.StringConverter
```

Interview note: the registry's mode is a policy, not a guarantee. It only checks schemas that are actually registered; a producer that bypasses the registry can still write incompatible bytes.

The full set is `BACKWARD`, `FORWARD`, `FULL`, `NONE`, and the `_TRANSITIVE` variants. A transitive mode compares a candidate against every registered version, not only the latest, so `BACKWARD_TRANSITIVE` keeps every historical reader working. Subjects are usually `<topic>-value` and `<topic>-key`, and compatibility is set per subject or as a global default.

Common mistake: setting BACKWARD and believing old consumers can read new data. BACKWARD protects new readers over old data. FORWARD protects old readers over new data. When both directions matter, use FULL.

## How do aliases work in Avro schema evolution?

`aliases` let a reader treat one name as another, which is how you rename a field or record without breaking existing data.

```json
{
  "type": "record",
  "name": "OrderCreated",
  "aliases": ["OrderEvent"],
  "fields": [
    {"name": "order_id", "type": "string"},
    {"name": "gross_amount", "type": "double", "aliases": ["amount"]}
  ]
}
```

Rules:
+ Aliases are applied only during resolution (read), so old files and old messages keep working.
+ Renaming a record means you should also update the subject name or keep the previous name as an alias.
+ Aliases also let a reader pull a field from a renamed record, but they do not fix type changes.
+ Adding `aliases` does not change the parsing canonical form, so a registry treats it as the same schema for parsing purposes but as a new version for resolution.

An alias is a reader-side name. During resolution the reader's aliases rewrite the writer's schema, so `gross_amount` with `"aliases": ["amount"]` reads a writer field named `amount`. Aliases also work on records, enums, and fixed types, and may be namespace-qualified. Adding an alias to the new reader makes a rename backward compatible. The rename is not forward compatible, because the old reader has no alias for the new name, unless the new field also has a default.

Common mistake: putting the alias on the writer schema or on the new field name. The alias belongs on the side that reads, and it points back at the old name.

## What are common Avro schema evolution pitfalls?

+ Adding a required field (no default) breaks backward compatibility for every existing reader.
+ Writing `null` into a union that does not declare `null` fails at serialization time, not at read time.
+ Removing `null` from `["null", "string"]` after data was written with nulls makes old data unreadable.
+ Using a bare `double` for money and later wanting `decimal` is a breaking type change; design money as decimal from day one.
+ Putting logically distinct events in one schema produces dozens of optional fields; prefer one topic/schema per event type with a shared envelope.
+ Relying on default `double`/`float` for IDs loses precision above 2^53.
+ Forgetting that a schema registry subject per topic means a rename of the record also changes the resolved schema everywhere.
+ Using `doc` or field descriptions as the only documentation; they are stripped from the canonical form and not enforced.

More traps: changing a decimal's `scale` or `precision` changes the value; removing an enum symbol makes old data unreadable; moving a record to a new namespace changes its full name; a union default that does not equal the first branch is rejected at parse time; and changing a field's type from `string` to `long` is not a legal promotion.

A default is applied only when the writer did not write the field at all. If the writer writes `null` for a field whose reader expects `string`, no default saves the read. That is why removing `null` from a union that old data used is a breaking change.

## Explain the Avro Sasl profile?

Basically, SASL offers a framework for authentication and security of network protocols. In Avro also we use SASL Profile for authentication and security purposes.

Avro RPC uses SASL for the handshake, and the common mechanisms are `PLAIN`, `DIGEST-MD5`, `GSSAPI` (Kerberos), and `ANONYMOUS` for trusted networks only. In a modern Kafka pipeline, SASL is configured on the Kafka broker/client side rather than in Avro itself:

```yaml
# Kafka Connect worker properties: SASL_SSL plus Avro serialization via the registry.
security.protocol: SASL_SSL
sasl.mechanism: SCRAM-SHA-512
sasl.jaas.config: >-
  org.apache.kafka.common.security.scram.ScramLoginModule required
  username="svc-cdc" password="${file:/opt/kafka/secrets/cdc.password}";
value.converter: io.confluent.connect.avro.AvroConverter
value.converter.schema.registry.url: https://schema-registry.example.com
value.converter.basic.auth.credentials.source: USER_INFO
value.converter.basic.auth.user.info: svc-cdc:${file:/opt/kafka/secrets/sr.password}
```

Avro RPC runs a handshake before any messages. The client sends the MD5 of its protocol; the server answers `match` when the two agree, or `both` when both protocols must be sent. Each request is a metadata map, the message name, and the encoded parameters. Each response is metadata plus a flag and either the encoded result or an error union.

SASL wraps that transport for authentication, with mechanisms `PLAIN`, `DIGEST-MD5`, `GSSAPI` (Kerberos), and `ANONYMOUS` for trusted networks only. Common mistake: treating SASL as schema security. It authenticates the connection, not the data contract.

## Why is Avro useful in streaming and CDC pipelines?

Avro is compact, schema-based, and well suited for record-oriented data. It is commonly used with Kafka because producers and consumers can exchange small binary messages while a schema registry manages compatibility. This helps pipelines evolve without every consumer breaking when a new optional field is added.

Example schema:

```json
{
  "type": "record",
  "name": "OrderEvent",
  "fields": [
    {"name": "order_id", "type": "string"},
    {"name": "amount", "type": ["null", "double"], "default": null},
    {"name": "event_time", "type": "string"}
  ]
}
```

The key interview point is that Avro stores data with a schema contract. This is safer than passing ungoverned JSON between teams when fields and types change over time.

In CDC specifically, Debezium emits a well-known envelope (before/after/op/source) and its `AvroConverter` registers that envelope in the registry:

```json
{
  "name": "orders-avro-connector",
  "config": {
    "connector.class": "io.debezium.connector.postgresql.PostgresConnector",
    "database.hostname": "pg-primary.internal",
    "database.port": "5432",
    "database.dbname": "orders",
    "table.include.list": "public.orders,public.order_items",
    "plugin.name": "pgoutput",
    "topic.prefix": "cdc",
    "key.converter": "io.confluent.connect.avro.AvroConverter",
    "key.converter.schema.registry.url": "https://schema-registry.example.com",
    "value.converter": "io.confluent.connect.avro.AvroConverter",
    "value.converter.schema.registry.url": "https://schema-registry.example.com",
    "decimal.handling.mode": "precise",
    "time.precision.mode": "adaptive_time_microseconds",
    "tombstones.on.delete": "true",
    "heartbeat.interval.ms": "10000"
  }
}
```

The Confluent wire format is one magic byte `0`, a 4-byte big-endian schema id, and then the Avro payload. The payload itself is schemaless; the id is the pointer. Consumers fetch the writer schema by id and resolve it with their reader schema, so a topic can evolve without every message carrying the full schema.

Common mistake: writing with a schema that was never registered, or reusing a serializer that cached a stale schema. The id in the message then points at the wrong contract, and consumers decode the wrong fields.

## When to use Avro?

Mainly, for two purposes, we use Avro, like:
+ Data serialization
+ RPC (Remote procedure call) protocol

Although, some key points are:
+ We are able to read the data from disk with applications written in other languages besides Java or the JVM.
+ Avro allows us to transfer data across a remote system without the overhead of Java serialization.
+ We use Avro when we need to store a large set of data on disk, as it conserves space.
+ By using Avro for RPC, we get better remote data transfer.

Choose Avro when:
+ Kafka is the backbone and a schema registry is available.
+ Row-oriented files matter more than columnar scan performance (Avro beats Parquet for full-row streaming and write-heavy landing zones).
+ You need a self-describing container file for archival or handoff.

Do not choose Avro when:
+ The workload is analytical and scan-heavy by a few columns: Parquet or ORC will be much faster and smaller.
+ Humans need to read the payload directly.
+ You need random access or column pruning.

Row versus column is the first decision. Avro stores whole records, so it favors full-row scans, streaming, and write-heavy landing zones. Parquet and ORC store columns, so they favor analytical scans of a few columns and compression tuned per column type. Schema behavior is the second decision: Avro, Protobuf, and Thrift carry or reference a schema, while JSON and CSV do not.

| Format | Layout | Schema | Best for | Weak spot |
|---|---|---|---|---|
| Avro | row, binary | writer schema or registry id | Kafka events, CDC, whole-row ETL | analytical column scans |
| Parquet | column, binary | embedded footer | lakehouse analytics, large scans | per-message streaming |
| ORC | column, binary | embedded footer | Hive-style analytics, ACID tables | ecosystem outside Hive |
| Protobuf | row, tagged | `.proto` or registry | gRPC APIs, polyglot services | needs code generation |
| JSON | row, text | none or JSON Schema | APIs, debugging, small volumes | size, no strict typing |

Common mistake: choosing one format for the whole platform. The usual answer is Avro on the stream and Parquet in the lake, with JSON kept at the edges for debugging.

## Who is intended audience to Learn Avro?

Those people who want to learn the basics of Big Data Analytics by using the Hadoop Framework, and also those who aspire to become a successful Hadoop developer, can go for Avro. Further, those aspirants who want to use Avro for data serialization and deserialization can also learn Avro. In practice the audience today is mostly streaming and data-platform engineers: Kafka producers/consumers, CDC pipeline owners, Spark and Flink developers, and platform teams governing schemas.

As a working data engineer, treat Avro as the topic you need when you own Kafka topics, CDC feeds, or landing-zone files. Interview questions about Avro are usually about evolution rules and registry policy, not the Java API.

## What are prerequisites to learn Avro?

Those who want to learn Avro must know Hadoop's architecture and APIs before learning Avro. Also, they must know Java with experience in writing basic applications before going for Avro. For a modern streaming role, the more useful prerequisites are: JSON, a Kafka producer/consumer, one of the Avro libraries (`fastavro`, `avro-python3`, or Java `avro`), and the concept of a schema registry.

Know varints and the idea of a schema registry before an Avro interview. A short way to build intuition is to encode and decode your own varints, and to compare two schema dicts for compatibility in Python.

## What is the advantage of Hadoop Over Java Serialization?

As with the help of the Writable objects, Hadoop Writable-based serialization is able to reduce object-creation overhead, which is not possible with the Java native serialization framework; that is why using the Hadoop one is an advantage.

Comparison:

+ Java serialization writes class names, field metadata, and object graph information with every stream, which is large and not portable.
+ Hadoop `Writable` is compact and reusable (`write`/`readFields` into the same object), but it is Java-only and tightly coupled to a specific class.
+ Avro goes one step further: it is compact, cross-language, and schema-governed, so Hive, Spark, and Python can all read the same files.

The Avro advantage over `Writable` is portability plus evolution. The schema and the file format are language-neutral, and a reader can resolve a different schema from the writer. `Writable` gives speed and low object churn inside the JVM only.

## What are the disadvantages of Hadoop Serialization?

The only disadvantage of Hadoop Serialization is that the Writable and Sequence files have only a Java API. Hence, to solve this issue, Avro comes in the picture.

A simple Avro file written by Python and read by Spark demonstrates the portability that `Writable` lacks:

In [ ]:
# ILLUSTRATIVE - requires fastavro: install to run, or read on.
from fastavro import writer, reader

schema = {
    "type": "record",
    "name": "Metric",
    "fields": [
        {"name": "name", "type": "string"},
        {"name": "value", "type": "double"},
        {"name": "ts", "type": "long"},
    ],
}
writer(open("metrics.avro", "wb"), schema, [{"name": "cpu", "value": 0.42, "ts": 1791138182000}])

for row in reader(open("metrics.avro", "rb")):
    print(row)

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# The same file, read by Spark.
metrics = spark.read.format("avro").load("file:///data/metrics.avro")
metrics.show()

The Java-only API of `Writable` is why cross-language pipelines move to Avro or Parquet. Common mistake: assuming Avro replaces `Writable` for shuffle. The shuffle format is internal to the engine; Avro is for interchange and storage between systems and languages.

## How do you design Avro schemas and keep them compatible?

Designing an Avro schema is designing a contract you will live with for years. The rules below cover optional fields, unions, names, compatibility policy, storage choices, and when Avro is the wrong tool.

## Schema design for evolution

Make every field after the first release optional with a default. Declare an optional field as `["null", T]` with `"default": null`. Use a plain type only when the field is truly required for every event. Prefer a record over a union of records: unions grow combinatorially and every consumer must handle every branch. Use `aliases` for renames. Keep names and namespaces stable, because a namespace change renames the type.

v1:

```json
{
  "type": "record",
  "name": "Order",
  "namespace": "com.example.orders",
  "fields": [
    {"name": "order_id", "type": "string"},
    {"name": "status", "type": "string"}
  ]
}
```

v2 adds a defaulted field, an optional field, and a renamed field with an alias:

```json
{
  "type": "record",
  "name": "Order",
  "namespace": "com.example.orders",
  "fields": [
    {"name": "order_id", "type": "string"},
    {"name": "state", "type": "string", "aliases": ["status"]},
    {"name": "currency", "type": "string", "default": "USD"},
    {"name": "coupon", "type": ["null", "string"], "default": null}
  ]
}
```

Common mistake: making a field required because it is always present today. Messages already in the log have no value for it, so every new reader fails on old data unless the field has a default.

## Compatibility policy and the registry workflow

Pick the mode from the upgrade order. Consumers first means BACKWARD. Producers first means FORWARD. Neither order is guaranteed means FULL. Add `_TRANSITIVE` when every historical version must stay readable. Choose one subject naming strategy, such as TopicName or RecordName, and keep it, because switching strategies resets the compatibility history of a subject.

```bash
# Register a candidate schema as a new version under the topic-value subject.
jq -Rs '{schema: .}' order_v2.avsc > payload.json
curl -s -X POST -H "Content-Type: application/vnd.schemaregistry.v1+json" \
  --data @payload.json \
  https://schema-registry.example.com/subjects/orders.events.v1-value/versions

# CI gate: test compatibility without registering anything.
curl -s -X POST -H "Content-Type: application/vnd.schemaregistry.v1+json" \
  --data @payload.json \
  https://schema-registry.example.com/compatibility/subjects/orders.events.v1-value/versions/latest

# Pin the policy for the subject, and check every historical version.
curl -s -X PUT -H "Content-Type: application/vnd.schemaregistry.v1+json" \
  --data '{"compatibility": "BACKWARD_TRANSITIVE"}' \
  https://schema-registry.example.com/config/orders.events.v1-value
```

Common mistake: changing the global compatibility default for one team's convenience, or deleting and re-registering under the same subject. Both make the registry's checks meaningless for everyone else.

## Encoding and storage choices

Container files record `avro.codec` and compress each block independently. `null` is fastest, `deflate` is universal, `snappy` is fast with a modest ratio, and `zstandard` usually gives the best ratio per unit of CPU. `sync_interval` sets how often the 16-byte sync marker repeats; a smaller interval adds seek points and per-block overhead.

Avro also defines a JSON encoding of data for the same schemas. It is useful for debugging and for languages without a binary library, but it is several times larger and is not for production volume. Common mistake: using very large blocks. One block is the unit of compression and of a task's read, so huge blocks reduce parallelism.

## When do you choose Avro vs Parquet vs Protobuf?

Decision rules:
+ Avro when records are small, writes are constant, and many consumers must evolve independently.
+ Parquet or ORC when queries scan a few columns across many rows.
+ Protobuf when services talk over gRPC and share `.proto` files.
+ JSON at the edge, where humans and browsers are the readers.

Common mistake: using Avro for a warehouse-style scan table, or Parquet for a low-latency per-message stream. Match the format to the dominant read pattern, not to the team's habit.

## Real world use cases for Apache Avro

### 1. E-commerce order events on Kafka

Business context: the checkout service publishes an `OrderEvent` for every state change, and five downstream teams (fulfilment, fraud, analytics, notifications, data lake) consume it. Volume: 40 million events/day at peak 3,000 events/second, roughly 120 bytes per event.

Architecture: producer serializes with `AvroSerializer`; the schema lives in a registry under subject `orders.events.v1-value` with `BACKWARD` compatibility; consumers use `AvroDeserializer`; a Kafka Connect S3 sink writes Avro object container files to a raw zone.

```yaml
name: orders-avro-s3-sink
config:
  connector.class: io.confluent.connect.s3.S3SinkConnector
  topics: orders.events.v1
  tasks.max: "6"
  s3.bucket.name: company-raw
  s3.part.size: "67108864"
  flush.size: "50000"
  storage.class: io.confluent.connect.s3.storage.S3Storage
  format.class: io.confluent.connect.s3.format.avro.AvroFormat
  value.converter: io.confluent.connect.avro.AvroConverter
  value.converter.schema.registry.url: https://schema-registry.example.com
```

Why this design: Avro keeps per-event overhead low at thousands of messages per second, the registry blocks a producer from shipping a required field that would break the five consumers, and Avro container files remain splittable so the nightly Spark job over the raw zone parallelizes cleanly.

### 2. Fintech ledger CDC into a lakehouse

Business context: a payments platform must replicate its PostgreSQL ledger to an S3 warehouse with exact amounts and audit-ready ordering. Volume: 12 million ledger rows, 8,000 updates/second during settlement windows, decimal amounts that must not lose precision.

Architecture: Debezium reads the Postgres WAL and emits the Avro envelope (`before`, `after`, `op`, `source.lsn`, `source.txId`); the topic is compacted by primary key; Spark Structured Streaming deduplicates on `(ledger_id)` ordered by `source.lsn` and merges into Iceberg.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F
from pyspark.sql.window import Window

raw = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka-1:9092")
    .option("subscribe", "cdc.public.ledger")
    .option("startingOffsets", "earliest")
    .load()
)

envelope = (
    raw.select(F.from_avro(F.col("value"), ledger_envelope_schema).alias("e"))
    .select("e.after.*", "e.op", "e.source.lsn", "e.source.ts_ms")
)

window = Window.partitionBy("ledger_id").orderBy(F.col("lsn").desc(), F.col("ts_ms").desc())
deduped = (
    envelope.withColumn("rn", F.row_number().over(window))
    .where(F.col("rn") == 1)
    .drop("rn")
)

Why this design: Avro's `decimal(18,2)` logical type preserves exact money values, `source.lsn` gives a monotonic ordering key so replays are deterministic, and the raw Avro topic is the immutable audit trail while the Iceberg table is the queryable current state.

### 3. Multi-tenant SaaS event ingestion

Business context: a B2B SaaS platform ingests product telemetry for 2,000 tenants, each with slightly different custom properties. Volume: 300 million events/day, and tenant schemas change independently.

Architecture: a shared `EventEnvelope` Avro schema carries `tenant_id`, `event_name`, `occurred_at`, and a `map<string,string>` for tenant-specific properties; per-tenant subjects (`telemetry.tenant-42.properties-value`) version the custom dimension schema; a generic Avro consumer decodes the envelope dynamically.

In [ ]:
# ILLUSTRATIVE - requires fastavro: install to run, or read on.
from fastavro import parse_schema, schemaless_reader
import io

envelope_schema = parse_schema({
    "type": "record",
    "name": "EventEnvelope",
    "namespace": "com.example.telemetry",
    "fields": [
        {"name": "tenant_id", "type": "string"},
        {"name": "event_name", "type": "string"},
        {"name": "occurred_at", "type": {"type": "long", "logicalType": "timestamp-millis"}},
        {"name": "props", "type": {"type": "map", "values": "string"}, "default": {}},
    ],
})

def decode(payload: bytes) -> dict:
    return schemaless_reader(io.BytesIO(payload), envelope_schema)

Why this design: one stable core schema plus a free-form map avoids 2,000 registered schemas for the hot path, while a per-tenant subject still governs the optional typed extensions. Avro's `map` is string-keyed, which is exactly what tenant properties are.

### 4. GDPR-safe customer events with tokenized PII

Business context: a European retailer must keep analytics events while ensuring raw email and phone never land in the lake. Volume: 5 million events/day, 90-day retention on raw tokens, 3-year retention on aggregates.

Architecture: the ingestion service stores PII in a vault and replaces it with a deterministic token before serializing. The Avro schema marks the token field and carries no raw PII; a `logicalType` uuid keeps tokens compact.

In [ ]:
# ILLUSTRATIVE - requires fastavro: install to run, or read on.
from fastavro import writer

gdpr_events = [
    {
        "customer_token": "3f9c1b6e-6b1c-4f5e-9c1a-2f4d7e8a0b12",
        "event_type": "checkout_started",
        "occurred_at": 1791138182000,
    }
]
writer(open("gdpr_events.avro", "wb"), gdpr_schema, gdpr_events)

Why this design: because the PII never enters the Avro payload, a deletion request only needs the vault mapping removed and the token rows in the lake become meaningless. The schema documents the contract so a producer cannot quietly add an `email` field without a registry compatibility review.

### 5. Cross-language ML feature logging

Business context: an ML platform logs features and predictions for training-set reconstruction and drift monitoring. Volume: 2 TB/day compressed, read back months later by Python, Scala, and Java jobs.

Architecture: online services (Go, Java) write Avro container files; offline Spark and Python read the same files; the schema is stored alongside the model artifact with the git commit SHA so a model reproduces against the exact feature contract.

```json
{
  "type": "record",
  "name": "FeatureLog",
  "namespace": "com.example.ml",
  "fields": [
    {"name": "request_id", "type": "string"},
    {"name": "model_version", "type": "string"},
    {"name": "features", "type": {"type": "map", "values": "double"}},
    {"name": "prediction", "type": "double"},
    {"name": "logged_at", "type": {"type": "long", "logicalType": "timestamp-millis"}}
  ]
}
```

Why this design: Avro container files are self-describing and splittable, so a six-month-old file still decodes without the service that wrote it; the schema-in-git pairing makes training data reproducible; and the map keeps the feature set flexible without a new schema per model.

## Avro exercises

These exercises use only the Python standard library and model Avro encoding and compatibility with small, runnable calculations. Each one names the task, gives a hint, states the expected result, then prints and asserts the answer. Run them with the repository virtual environment: `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.

The sizes are spec-based, not measurements of a particular library. The value of each exercise is knowing which rule to apply and which direction a change moves compatibility.

### Try it 1. Compute the varint byte size of an integer

**Task.** Compute the Avro encoded size for `0`, `-1`, `63`, `64`, `8191`, `8192`, and `2**31 - 1`.

**Hint.** Avro zig-zags the value first: `zigzag(n) = (n << 1) ^ (n >> 63)`. The byte count is `ceil(bit_length / 7)`, with a minimum of one byte.

**Expected result.** 0 and -1 take 1 byte, 63 takes 1, 64 takes 2, 8191 takes 2, 8192 takes 3, and 2**31 - 1 takes 5.

In [ ]:
def zigzag(n):
    return (n << 1) ^ (n >> 63)


def varint_size(n):
    z = zigzag(n)
    return max(1, (z.bit_length() + 6) // 7)


sizes = {n: varint_size(n) for n in [0, -1, 63, 64, 8191, 8192, 2**31 - 1]}
for value, size in sizes.items():
    print(f"{value:>12} -> {size} byte(s)")

assert sizes[0] == 1
assert sizes[-1] == 1
assert sizes[63] == 1
assert sizes[64] == 2
assert sizes[8191] == 2
assert sizes[8192] == 3
assert sizes[2**31 - 1] == 5

### Try it 2. Decide if a schema change is BACKWARD or FORWARD compatible

**Task.** Given an old and a new record schema dict, decide whether the change is `FULL`, `BACKWARD`, `FORWARD`, or `NEITHER`. Test three changes: adding an optional field, adding a required field, and dropping a required field.

**Hint.** BACKWARD means the new schema reads old data, so a field added in the new schema needs a `default`. FORWARD means the old schema reads new data, so a field removed from the new schema needs a `default` in the old schema.

**Expected result.** Adding an optional field is FULL, adding a required field is FORWARD, and dropping a required field is BACKWARD.

In [ ]:
def fields(schema):
    return {f["name"]: f for f in schema["fields"]}


def backward(old, new):
    old_fields = fields(old)
    return all(name in old_fields or "default" in field
               for name, field in fields(new).items())


def forward(old, new):
    new_fields = fields(new)
    return all(name in new_fields or "default" in field
               for name, field in fields(old).items())


def verdict(old, new):
    b, f = backward(old, new), forward(old, new)
    if b and f:
        return "FULL"
    if b:
        return "BACKWARD"
    if f:
        return "FORWARD"
    return "NEITHER"


base = {"type": "record", "name": "Order", "fields": [
    {"name": "order_id", "type": "string"},
    {"name": "status", "type": "string"}]}
changes = {
    "add optional": {"type": "record", "name": "Order", "fields": [
        {"name": "order_id", "type": "string"},
        {"name": "status", "type": "string"},
        {"name": "currency", "type": "string", "default": "USD"}]},
    "add required": {"type": "record", "name": "Order", "fields": [
        {"name": "order_id", "type": "string"},
        {"name": "status", "type": "string"},
        {"name": "currency", "type": "string"}]},
    "drop required": {"type": "record", "name": "Order", "fields": [
        {"name": "order_id", "type": "string"}]},
}

for label, new in changes.items():
    print(f"{label:14} -> {verdict(base, new)}")

assert verdict(base, changes["add optional"]) == "FULL"
assert verdict(base, changes["add required"]) == "FORWARD"
assert verdict(base, changes["drop required"]) == "BACKWARD"

### Try it 3. Compute a record size from its field types

**Task.** For the record `{"order_id": "O-1001", "quantity": 3, "price": 19.99, "active": True}`, compute each field's encoded size and the total.

**Hint.** `string` is a length varint plus the UTF-8 bytes; `int` and `long` are varints; `double` is 8 bytes; `boolean` is 1 byte; `null` is 0 bytes.

**Expected result.** order_id is 7 bytes, quantity is 1, price is 8, active is 1, so the total is 17 bytes.

In [ ]:
def zigzag(n):
    return (n << 1) ^ (n >> 63)


def varint_size(n):
    z = zigzag(n)
    return max(1, (z.bit_length() + 6) // 7)


def size_of(type_name, value):
    if type_name == "null":
        return 0
    if type_name == "boolean":
        return 1
    if type_name in ("int", "long"):
        return varint_size(value)
    if type_name == "float":
        return 4
    if type_name == "double":
        return 8
    if type_name == "string":
        raw = value.encode()
        return varint_size(len(raw)) + len(raw)
    raise ValueError(type_name)


record = {"order_id": ("string", "O-1001"), "quantity": ("int", 3),
          "price": ("double", 19.99), "active": ("boolean", True)}
total = 0
for name, (type_name, value) in record.items():
    n = size_of(type_name, value)
    total += n
    print(f"{name:9} {type_name:8} {n:>3} bytes")
print("record payload:", total, "bytes")

assert size_of("string", "O-1001") == 7
assert size_of("int", 3) == 1
assert size_of("double", 19.99) == 8
assert total == 17

### Try it 4. Validate a schema dict structurally

**Task.** Write a validator that walks a schema and reports missing required keys. It must handle `record`, `enum`, `array`, `map`, and `fixed`, plus unions. Run it on a valid schema and on one that is missing a record name and a field type.

**Hint.** A record needs `name` and a `fields` list; every field needs `name` and `type`; an enum needs `name` and `symbols`; an array needs `items`; a map needs `values`; a fixed needs `name` and `size`; a union cannot contain another union.

**Expected result.** The valid schema reports no errors. The broken schema reports exactly two: the record has no name, and its first field has no type.

In [ ]:
def validate(schema, path="$"):
    errors = []
    if isinstance(schema, list):
        if not schema:
            errors.append(f"{path}: union must have at least one branch")
        if any(isinstance(branch, list) for branch in schema):
            errors.append(f"{path}: union cannot contain another union")
        for i, branch in enumerate(schema):
            errors.extend(validate(branch, f"{path}[{i}]"))
        return errors
    if isinstance(schema, str):
        return errors
    if not isinstance(schema, dict):
        return [f"{path}: schema must be a string, object, or array"]
    kind = schema.get("type")
    if kind == "record":
        if "name" not in schema:
            errors.append(f"{path}: record needs a name")
        if not isinstance(schema.get("fields"), list):
            errors.append(f"{path}: record needs a fields array")
        else:
            for i, field in enumerate(schema["fields"]):
                if "name" not in field or "type" not in field:
                    errors.append(f"{path}.fields[{i}]: field needs name and type")
    elif kind == "enum":
        if "name" not in schema or not isinstance(schema.get("symbols"), list):
            errors.append(f"{path}: enum needs a name and symbols")
    elif kind == "array":
        if "items" not in schema:
            errors.append(f"{path}: array needs items")
    elif kind == "map":
        if "values" not in schema:
            errors.append(f"{path}: map needs values")
    elif kind == "fixed":
        if "name" not in schema or "size" not in schema:
            errors.append(f"{path}: fixed needs a name and size")
    return errors


good = {"type": "record", "name": "Order", "fields": [
    {"name": "order_id", "type": "string"},
    {"name": "tags", "type": {"type": "array", "items": "string"}},
    {"name": "props", "type": {"type": "map", "values": "string"}},
    {"name": "state", "type": {"type": "enum", "name": "State", "symbols": ["NEW", "PAID"]}},
    {"name": "checksum", "type": {"type": "fixed", "name": "Md5", "size": 16}},
]}
bad = {"type": "record", "fields": [{"name": "order_id"}]}

print("good:", validate(good))
print("bad: ", validate(bad))

assert validate(good) == []
assert len(validate(bad)) == 2

### Try it 5. Detect a breaking change: a dropped field without a default

**Task.** Given an old and a new schema, list the fields that were dropped from the new schema without a default. Those fields break FORWARD compatibility.

**Hint.** Collect the new field names, then return every old field that is absent from the new schema and has no `default` key.

**Expected result.** Only `status` is reported; `currency` is dropped but has a default, so it is safe.

In [ ]:
def dropped_required(old, new):
    new_names = {f["name"] for f in new["fields"]}
    return [f["name"] for f in old["fields"]
            if f["name"] not in new_names and "default" not in f]


old = {"type": "record", "name": "Order", "fields": [
    {"name": "order_id", "type": "string"},
    {"name": "status", "type": "string"},
    {"name": "currency", "type": "string", "default": "USD"}]}
new = {"type": "record", "name": "Order", "fields": [
    {"name": "order_id", "type": "string"}]}

breaks = dropped_required(old, new)
print("dropped without default:", breaks)

assert breaks == ["status"]

### Try it 6. Compare storage for row versus column layout

**Task.** A table has columns `id` (8 bytes), `user` (16), `payload` (64), and `flags` (4), with 1,000,000 rows. A query touches a column-access profile of id 100%, user 50%, payload 5%, flags 10%. Compute the bytes read by a row layout and by a column layout.

**Hint.** A row layout reads the full width of every row. A column layout reads `column_bytes * rows * access_fraction` for each accessed column.

**Expected result.** The row layout reads 92,000,000 bytes and the column layout about 19,600,000, which is 0.213 of the row bytes.

In [ ]:
COLUMNS = {"id": 8, "user": 16, "payload": 64, "flags": 4}
ROWS = 1_000_000
profile = {"id": 1.0, "user": 0.5, "payload": 0.05, "flags": 0.1}

row_bytes = sum(COLUMNS.values()) * ROWS
col_bytes = sum(COLUMNS[c] * ROWS * p for c, p in profile.items())
ratio = col_bytes / row_bytes

print(f"row layout:    {row_bytes:,} bytes")
print(f"column layout: {col_bytes:,.0f} bytes ({ratio:.3f} of row)")

assert row_bytes == 92_000_000
assert round(col_bytes) == 19_600_000
assert round(ratio, 3) == 0.213

### Try it 7. Plan registry subjects for a topic naming scheme

**Task.** For the topics `orders.events.v1` and `payments.ledger.v2`, list the key and value subjects under the TopicName strategy. Sort the result.

**Hint.** A subject is `<topic>-key` or `<topic>-value`. Build the cross product of topics and suffixes, then sort.

**Expected result.** Four subjects: orders.events.v1-key, orders.events.v1-value, payments.ledger.v2-key, payments.ledger.v2-value.

In [ ]:
topics = ["orders.events.v1", "payments.ledger.v2"]
subjects = sorted([f"{topic}-key" for topic in topics]
                  + [f"{topic}-value" for topic in topics])

for subject in subjects:
    print(subject)

assert len(subjects) == 4
assert "orders.events.v1-value" in subjects
assert "payments.ledger.v2-key" in subjects

### Try it 8. Encode and decode a zig-zag varint by hand

**Task.** Implement `zigzag_encode`, `zigzag_decode`, `varint_bytes`, and `varint_decode`. Round-trip several values, including a large positive and negative one, and print the bytes for 8192.

**Hint.** Zig-zag maps signed to unsigned: `(n << 1) ^ (n >> 63)`; decode with `(z >> 1) ^ -(z & 1)`. Base-128 groups 7 bits per byte, and the high bit marks continuation.

**Expected result.** 8192 encodes as the 3 bytes `80 80 01`, and every tested value round-trips through both layers.

In [ ]:
def zigzag_encode(n):
    return (n << 1) ^ (n >> 63)


def zigzag_decode(z):
    return (z >> 1) ^ -(z & 1)


def varint_bytes(z):
    out = bytearray()
    while True:
        byte = z & 0x7F
        z >>= 7
        if z:
            out.append(byte | 0x80)
        else:
            out.append(byte)
            return bytes(out)


def varint_decode(data):
    z = 0
    for i, byte in enumerate(data):
        z |= (byte & 0x7F) << (7 * i)
        if not byte & 0x80:
            return z, i + 1
    raise ValueError("truncated varint")


for n in [0, -1, 1, -64, 64, 8192, -(2**40), 2**40]:
    encoded = varint_bytes(zigzag_encode(n))
    decoded, used = varint_decode(encoded)
    assert zigzag_decode(decoded) == n
    assert used == len(encoded)

print("8192 ->", varint_bytes(zigzag_encode(8192)).hex())
assert varint_bytes(zigzag_encode(8192)) == b"\x80\x80\x01"